# 05 — ORESTAR City Council fundraising profiles (2026)

Candidate fundraising profiles for **Portland City Council** using the cleaned
ORESTAR source for 2026.

The profile representation is kept identical across years:

- total contribution amount and contribution-record count;
- mean, median, minimum, maximum, and standard deviation;
- cash vs. in-kind contributions;
- Micro / Small / Medium / Large / Mega bins;
- amount and count shares by bin.

Confirmed candidate linkage is attached when available. Source-only candidates
are retained rather than silently dropped.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find repository root. "
        "Expected pyproject.toml in the current directory or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    CLEAN,
    PROCESSED,
    fundraising_processed_dir,
    orestar_file_audit_path,
    orestar_transactions_path,
    spending_processed_dir,
)

print("ROOT:", ROOT)

YEAR = 2026
CONTEST = "city_council"

TRANSACTIONS_PATH = orestar_transactions_path(
    YEAR,
    CONTEST,
)

AUDIT_PATH = orestar_file_audit_path(
    YEAR,
    CONTEST,
)

CROSSWALK_PATH = (
    PROCESSED
    / "master"
    / f"candidate_source_crosswalk_{YEAR}.csv"
)

OUTPUT_DIR = fundraising_processed_dir(
    YEAR,
    CONTEST,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Transactions:", TRANSACTIONS_PATH)
print("Audit:", AUDIT_PATH)
print("Crosswalk exists:", CROSSWALK_PATH.exists())
print("Output:", OUTPUT_DIR)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis
Transactions: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2026/city_council/transactions.csv
Audit: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2026/city_council/file_audit.csv
Crosswalk exists: False
Output: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/fundraising/2026/city_council


## 2. Load and validate cleaned City Council ORESTAR

In [2]:
def to_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    low_memory=False,
)

required = {
    "year",
    "contest_type",
    "office",
    "district",
    "source_file_stem",
    "sub_type",
    "is_reported_contribution",
    "reported_contribution_amount",
}

missing = sorted(
    required
    - set(transactions.columns)
)

if missing:
    raise ValueError(
        f"Missing clean ORESTAR columns: {missing}"
    )

contest_values = set(
    transactions["contest_type"]
    .dropna()
    .astype(str)
    .unique()
)

if contest_values != {CONTEST}:
    raise ValueError(
        f"Expected only {CONTEST}; found {sorted(contest_values)}"
    )

print("Rows:", f"{len(transactions):,}")
print("Districts:", sorted(
    transactions["district"]
    .dropna()
    .unique()
    .tolist()
))

display(
    transactions["sub_type"]
    .value_counts(dropna=False)
    .to_frame("rows")
)


Rows: 12,414
Districts: [3, 4]


,rows
sub_type,
Cash Contribution,8231
Cash Expenditure,3616
Personal Expenditure for Reimbursement,174
In-Kind Contribution,110
Return or Refund of Contribution,107
Items Sold at Fair Market Value,55
Account Payable,41
Loan Received (Non-Exempt),23
Refunds and Rebates,19


## 3. Keep positive reported contributions

The cleaner already defines `Cash Contribution` and `In-Kind Contribution` as
positive reported contributions and excludes refunds.


In [3]:
transactions["reported_contribution_amount"] = pd.to_numeric(
    transactions["reported_contribution_amount"],
    errors="coerce",
)

contributions = transactions.loc[
    to_bool(
        transactions["is_reported_contribution"]
    )
    & transactions["reported_contribution_amount"].notna()
    & transactions["reported_contribution_amount"].gt(0)
].copy()

contributions["amount"] = (
    contributions["reported_contribution_amount"]
)

print("Contribution records:", f"{len(contributions):,}")
print("Total amount:", f"${contributions['amount'].sum():,.2f}")


Contribution records: 8,341
Total amount: $3,299,138.75


## 4. File audit

In [4]:
if AUDIT_PATH.exists():
    audit = pd.read_csv(
        AUDIT_PATH,
        low_memory=False,
    )

    duplicate_exports = audit.loc[
        to_bool(
            audit["is_exact_duplicate_export"]
        )
    ].copy()

    print(
        "Exact duplicate export rows:",
        len(duplicate_exports),
    )

    if len(duplicate_exports):
        display(
            duplicate_exports[
                [
                    "district",
                    "source_file",
                    "source_file_hash",
                    "same_hash_file_count",
                ]
            ]
        )
else:
    print("No file audit found.")


Exact duplicate export rows: 0


## 5. Candidate identity

When a confirmed ORESTAR crosswalk exists, use the official candidate key.
Otherwise preserve a stable source-level profile key.


In [5]:
def attach_orestar_identity(data: pd.DataFrame, crosswalk_path: Path) -> pd.DataFrame:
    frame = data.copy()

    frame["source_candidate_name"] = (
        frame["source_file_stem"]
        .astype("string")
        .str.strip()
    )

    frame["canonical_candidate"] = pd.NA
    frame["candidate_key"] = pd.NA
    frame["linkage_status"] = "source_only"

    if crosswalk_path.exists():
        crosswalk = pd.read_csv(
            crosswalk_path,
            low_memory=False,
        )

        needed = {
            "source",
            "classification",
            "year",
            "district",
            "source_candidate_name",
            "suggested_candidate",
            "suggested_candidate_key",
        }

        if needed.issubset(crosswalk.columns):
            matched = (
                crosswalk.loc[
                    crosswalk["source"].eq("orestar")
                    & crosswalk["classification"].eq("match"),
                    [
                        "year",
                        "district",
                        "source_candidate_name",
                        "suggested_candidate",
                        "suggested_candidate_key",
                    ],
                ]
                .rename(
                    columns={
                        "suggested_candidate": "_canonical_candidate",
                        "suggested_candidate_key": "_candidate_key",
                    }
                )
                .drop_duplicates()
            )

            frame = frame.merge(
                matched,
                on=[
                    "year",
                    "district",
                    "source_candidate_name",
                ],
                how="left",
                validate="many_to_one",
            )

            frame["canonical_candidate"] = frame["_canonical_candidate"]
            frame["candidate_key"] = frame["_candidate_key"]

            frame["linkage_status"] = np.where(
                frame["candidate_key"].notna(),
                "matched_to_official_candidate",
                "unmatched_source_candidate",
            )

            frame = frame.drop(
                columns=[
                    "_canonical_candidate",
                    "_candidate_key",
                ]
            )

    frame["candidate"] = (
        frame["canonical_candidate"]
        .fillna(frame["source_candidate_name"])
    )

    source_key = (
        frame["year"].astype("Int64").astype(str)
        + "|"
        + frame["district"].astype("Int64").astype(str)
        + "|orestar|"
        + frame["source_candidate_name"]
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    frame["profile_key"] = (
        frame["candidate_key"]
        .fillna(source_key)
    )

    return frame

contributions = attach_orestar_identity(
    contributions,
    CROSSWALK_PATH,
)

identity_summary = (
    contributions[
        [
            "year",
            "district",
            "source_file_stem",
            "candidate",
            "candidate_key",
            "profile_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "district",
            "candidate",
        ]
    )
)

display(identity_summary)

print(
    "\nLinkage status:"
)
display(
    identity_summary["linkage_status"]
    .value_counts(dropna=False)
    .to_frame("candidate_source_rows")
)


,year,district,source_file_stem,candidate,candidate_key,profile_key,linkage_status
0,2026,3,Brown,Brown,<NA>,2026|3|orestar|brown,source_only
16,2026,3,Corcoran,Corcoran,<NA>,2026|3|orestar|corcoran,source_only
85,2026,3,Johnson,Johnson,<NA>,2026|3|orestar|johnson,source_only
105,2026,3,Koyama,Koyama,<NA>,2026|3|orestar|koyama,source_only
2149,2026,3,Leon,Leon,<NA>,2026|3|orestar|leon,source_only
2244,2026,3,Morillo,Morillo,<NA>,2026|3|orestar|morillo,source_only
3443,2026,3,Mullen,Mullen,<NA>,2026|3|orestar|mullen,source_only
3549,2026,3,Novick,Novick,<NA>,2026|3|orestar|novick,source_only
7144,2026,3,Sollitt,Sollitt,<NA>,2026|3|orestar|sollitt,source_only
7219,2026,3,Zimmerman,Zimmerman,<NA>,2026|3|orestar|zimmerman,source_only



Linkage status:


,candidate_source_rows
linkage_status,
source_only,18


## 6. Candidate fundraising summary

In [6]:
PROFILE_KEYS = [
    "year",
    "contest_type",
    "office",
    "district",
    "profile_key",
    "candidate",
]

candidate_summary = (
    contributions
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        total_amount=("amount", "sum"),
        total_contribution_count=("amount", "size"),
        average_contribution=("amount", "mean"),
        median_contribution=("amount", "median"),
        min_contribution=("amount", "min"),
        max_contribution=("amount", "max"),
        std_contribution=("amount", "std"),
    )
)

type_summary = (
    contributions
    .pivot_table(
        index=PROFILE_KEYS,
        columns="sub_type",
        values="amount",
        aggfunc=[
            "sum",
            "size",
        ],
        fill_value=0,
    )
)

type_summary.columns = [
    "_".join(
        str(x)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        for x in col
    )
    for col in type_summary.columns
]

type_summary = (
    type_summary
    .reset_index()
)

candidate_summary = candidate_summary.merge(
    type_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

identity_cols = (
    contributions[
        [
            "profile_key",
            "source_file_stem",
            "source_candidate_name",
            "canonical_candidate",
            "candidate_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates(
        subset=["profile_key"]
    )
)

candidate_summary = candidate_summary.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="one_to_one",
)

candidate_summary = candidate_summary.sort_values(
    [
        "district",
        "total_amount",
    ],
    ascending=[
        True,
        False,
    ],
)

display(candidate_summary)


,year,contest_type,office,district,profile_key,candidate,total_amount,total_contribution_count,average_contribution,median_contribution,min_contribution,max_contribution,std_contribution,sum_cash_contribution,sum_in_kind_contribution,size_cash_contribution,size_in_kind_contribution,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
7,2026,city_council,Portland City Council,3,2026|3|orestar|novick,Novick,1098871.61,2884,381.023443,250.0,3.00,40000.0,1176.607389,1083330.14,15541.47,2842,42,Novick,Novick,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|koyama,Koyama,456686.26,1098,415.925556,50.0,1.00,70936.0,3668.061831,448019.31,8666.95,1081,17,Koyama,Koyama,<NA>,<NA>,source_only
5,2026,city_council,Portland City Council,3,2026|3|orestar|morillo,Morillo,385654.34,616,626.062240,100.0,1.00,50000.0,4328.867397,381507.57,4146.77,605,11,Morillo,Morillo,<NA>,<NA>,source_only
6,2026,city_council,Portland City Council,3,2026|3|orestar|mullen,Mullen,11670.00,97,120.309278,50.0,5.00,1225.0,189.420298,11670.00,0.00,97,0,Mullen,Mullen,<NA>,<NA>,source_only
8,2026,city_council,Portland City Council,3,2026|3|orestar|sollitt,Sollitt,8620.00,35,246.285714,100.0,5.00,2083.0,456.118378,8620.00,0.00,35,0,Sollitt,Sollitt,<NA>,<NA>,source_only
9,2026,city_council,Portland City Council,3,2026|3|orestar|zimmerman,Zimmerman,8355.00,60,139.250000,32.5,10.00,875.0,253.914815,8355.00,0.00,60,0,Zimmerman,Zimmerman,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|leon,Leon,6716.00,63,106.603175,55.0,5.00,875.0,156.690628,5841.00,875.00,62,1,Leon,Leon,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,4095.00,30,136.500000,100.0,5.00,500.0,131.975376,4095.00,0.00,30,0,Corcoran,Corcoran,<NA>,<NA>,source_only
0,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,2365.00,14,168.928571,187.5,25.00,305.0,84.265573,2365.00,0.00,14,0,Brown,Brown,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,575.00,12,47.916667,25.0,5.00,100.0,42.611263,575.00,0.00,12,0,Johnson,Johnson,<NA>,<NA>,source_only


## 7. Contribution-size bins

In [7]:
BIN_LABELS = [
    "Micro",
    "Small",
    "Medium",
    "Large",
    "Mega",
]

BIN_EDGES = [
    -np.inf,
    25,
    100,
    250,
    1000,
    np.inf,
]

contributions["contribution_bin"] = pd.cut(
    contributions["amount"],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=True,
    ordered=True,
)

display(
    contributions["contribution_bin"]
    .value_counts(sort=False)
    .to_frame("records")
)


,records
contribution_bin,
Micro,1445
Small,2273
Medium,2522
Large,1898
Mega,203


## 8. Long fundraising profile

In [8]:
profile_long = (
    contributions
    .groupby(
        PROFILE_KEYS
        + [
            "contribution_bin",
        ],
        observed=False,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount=("amount", "sum"),
        contribution_count=("amount", "size"),
    )
)

profile_totals = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        profile_total_amount=("amount", "sum"),
        profile_total_contribution_count=(
            "contribution_count",
            "sum",
        ),
    )
)

profile_long = profile_long.merge(
    profile_totals,
    on=PROFILE_KEYS,
    how="left",
    validate="many_to_one",
)

profile_long["amount_share"] = (
    profile_long["amount"]
    / profile_long["profile_total_amount"]
)

profile_long["contribution_share"] = (
    profile_long["contribution_count"]
    / profile_long["profile_total_contribution_count"]
)

profile_long = profile_long.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="many_to_one",
)

display(profile_long.head(15))


,year,contest_type,office,district,profile_key,candidate,contribution_bin,amount,contribution_count,profile_total_amount,profile_total_contribution_count,amount_share,contribution_share,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,Micro,25.0,1,2365.0,14,0.010571,0.071429,Brown,Brown,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,Small,100.0,2,2365.0,14,0.042283,0.142857,Brown,Brown,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,Medium,1660.0,9,2365.0,14,0.701903,0.642857,Brown,Brown,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,Large,580.0,2,2365.0,14,0.245243,0.142857,Brown,Brown,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,Mega,0.0,0,2365.0,14,0.000000,0.000000,Brown,Brown,<NA>,<NA>,source_only
5,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Micro,145.0,8,4095.0,30,0.035409,0.266667,Corcoran,Corcoran,<NA>,<NA>,source_only
6,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Small,480.0,8,4095.0,30,0.117216,0.266667,Corcoran,Corcoran,<NA>,<NA>,source_only
7,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Medium,1875.0,10,4095.0,30,0.457875,0.333333,Corcoran,Corcoran,<NA>,<NA>,source_only
8,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Large,1595.0,4,4095.0,30,0.389499,0.133333,Corcoran,Corcoran,<NA>,<NA>,source_only
9,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Mega,0.0,0,4095.0,30,0.000000,0.000000,Corcoran,Corcoran,<NA>,<NA>,source_only


## 9. Validate profile shares

In [9]:
validation = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount_share_sum=("amount_share", "sum"),
        contribution_share_sum=("contribution_share", "sum"),
    )
)

validation["amount_share_ok"] = np.isclose(
    validation["amount_share_sum"],
    1.0,
)

validation["contribution_share_ok"] = np.isclose(
    validation["contribution_share_sum"],
    1.0,
)

print(
    "All amount-share profiles valid:",
    validation["amount_share_ok"].all(),
)

print(
    "All contribution-share profiles valid:",
    validation["contribution_share_ok"].all(),
)

display(
    validation.loc[
        ~validation["amount_share_ok"]
        | ~validation["contribution_share_ok"]
    ]
)


All amount-share profiles valid: True
All contribution-share profiles valid: True


,year,contest_type,office,district,profile_key,candidate,amount_share_sum,contribution_share_sum,amount_share_ok,contribution_share_ok


## 10. Wide fundraising profile

In [10]:
metrics = [
    "amount",
    "amount_share",
    "contribution_count",
    "contribution_share",
]

wide_parts = []

for metric in metrics:
    part = (
        profile_long
        .pivot(
            index=PROFILE_KEYS,
            columns="contribution_bin",
            values=metric,
        )
        .reindex(columns=BIN_LABELS)
        .fillna(0)
    )

    part.columns = [
        f"{metric}_{str(bin_name).lower()}"
        for bin_name in part.columns
    ]

    wide_parts.append(part)

profile_wide = pd.concat(
    wide_parts,
    axis=1,
).reset_index()

profile_wide = profile_wide.merge(
    candidate_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

print("Rows:", len(profile_wide))
print("Columns:", len(profile_wide.columns))
display(profile_wide.head())


Rows: 18
Columns: 42


,year,contest_type,office,district,profile_key,candidate,amount_micro,amount_small,amount_medium,amount_large,amount_mega,amount_share_micro,amount_share_small,amount_share_medium,amount_share_large,amount_share_mega,contribution_count_micro,contribution_count_small,contribution_count_medium,contribution_count_large,contribution_count_mega,contribution_share_micro,contribution_share_small,contribution_share_medium,contribution_share_large,contribution_share_mega,total_amount,total_contribution_count,average_contribution,median_contribution,min_contribution,max_contribution,std_contribution,sum_cash_contribution,sum_in_kind_contribution,size_cash_contribution,size_in_kind_contribution,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2026,city_council,Portland City Council,3,2026|3|orestar|brown,Brown,25.00,100.00,1660.00,580.00,0.00,0.010571,0.042283,0.701903,0.245243,0.000000,1,2,9,2,0,0.071429,0.142857,0.642857,0.142857,0.000000,2365.00,14,168.928571,187.5,25.0,305.0,84.265573,2365.00,0.00,14,0,Brown,Brown,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,145.00,480.00,1875.00,1595.00,0.00,0.035409,0.117216,0.457875,0.389499,0.000000,8,8,10,4,0,0.266667,0.266667,0.333333,0.133333,0.000000,4095.00,30,136.500000,100.0,5.0,500.0,131.975376,4095.00,0.00,30,0,Corcoran,Corcoran,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,100.00,475.00,0.00,0.00,0.00,0.173913,0.826087,0.000000,0.000000,0.000000,7,5,0,0,0,0.583333,0.416667,0.000000,0.000000,0.000000,575.00,12,47.916667,25.0,5.0,100.0,42.611263,575.00,0.00,12,0,Johnson,Johnson,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|koyama,Koyama,6666.15,21609.24,35968.33,65330.58,327111.96,0.014597,0.047317,0.078759,0.143054,0.716273,342,384,202,144,26,0.311475,0.349727,0.183971,0.131148,0.023679,456686.26,1098,415.925556,50.0,1.0,70936.0,3668.061831,448019.31,8666.95,1081,17,Koyama,Koyama,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|leon,Leon,359.00,1331.00,2978.00,2048.00,0.00,0.053454,0.198183,0.443419,0.304943,0.000000,20,23,17,3,0,0.317460,0.365079,0.269841,0.047619,0.000000,6716.00,63,106.603175,55.0,5.0,875.0,156.690628,5841.00,875.00,62,1,Leon,Leon,<NA>,<NA>,source_only


## 11. Quick descriptive view

In [11]:
district_summary = (
    candidate_summary
    .groupby(
        [
            "year",
            "district",
        ],
        as_index=False,
    )
    .agg(
        candidate_source_rows=("profile_key", "nunique"),
        total_fundraising=("total_amount", "sum"),
        median_candidate_fundraising=("total_amount", "median"),
        min_candidate_fundraising=("total_amount", "min"),
        max_candidate_fundraising=("total_amount", "max"),
    )
)

display(district_summary)


,year,district,candidate_source_rows,total_fundraising,median_candidate_fundraising,min_candidate_fundraising,max_candidate_fundraising
0,2026,3,10,1983608.21,8487.500,575.0,1098871.61
1,2026,4,8,1315530.54,150415.485,600.0,340410.45


## 12. Export

In [12]:
summary_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_summary.csv"
)

long_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_profiles_long.csv"
)

wide_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_profiles_wide.csv"
)

candidate_summary.to_csv(
    summary_path,
    index=False,
)

profile_long.to_csv(
    long_path,
    index=False,
)

profile_wide.to_csv(
    wide_path,
    index=False,
)

print("SAVED", summary_path)
print("SAVED", long_path)
print("SAVED", wide_path)


SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/fundraising/2026/city_council/orestar_candidate_fundraising_summary.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/fundraising/2026/city_council/orestar_candidate_fundraising_profiles_long.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/fundraising/2026/city_council/orestar_candidate_fundraising_profiles_wide.csv
